In [21]:
import numpy as np
import pandas as pd
import requests

In [ ]:
class CninfoAPI:
    def __init__(self, key="***REMOVED***", secret="***REMOVED***", is_testAPI=False) -> None:
        if is_testAPI:
            self.ACCESS_KEY = "***REMOVED***"
            self.ACCESS_SECRET = "***REMOVED***"
            self.BASE_URL = "http://webapitest.cninfo.com.cn"
        else:
            self.ACCESS_KEY = key
            self.ACCESS_SECRET = secret
            self.BASE_URL = "http://webapi.cninfo.com.cn"
        
        request_for_token = requests.post(self.BASE_URL + "/api-cloud-platform/oauth2/token", data={
            "grant_type": "client_credentials",
            "client_id": self.ACCESS_KEY,
            "client_secret": self.ACCESS_SECRET
        })
        if request_for_token.status_code == requests.codes.ok:
            res_dict = request_for_token.json()
            self.token = res_dict.get('access_token')
        else:
            self.token = ""


    def getStockSeq(self, scode="000001", sdate="2022-01-13", edate="2022-01-25"):
        STOCK_API = "/api/stock/p_stock2402"
        stock_req = requests.get(self.BASE_URL+STOCK_API, params={
            "scode": scode, 
            "sdate": sdate.replace("-", ""), 
            "edate": edate.replace("-", ""), 
            "access_token": self.token
        })
        print(stock_req.json())
        if stock_req.status_code == requests.codes.ok:
            records = stock_req.json().get('records')
        else:
            records = {
                "code": stock_req.json().get('resultcode'),
                "msg": "cninfo接口错误：" + stock_req.json().get('resultmsg')
            }
        
        return records


    def getIndexSeq(self, scode="000300", sdate="2022-01-13", edate="2022-01-25"):
        INDEX_API = "/api/index/p_index2905"
        index_req = requests.get(self.BASE_URL+INDEX_API, params={
            "scode": scode, 
            "sdate": sdate.replace("-", ""), 
            "edate": edate.replace("-", ""), 
            "access_token": self.token
        })

        if index_req.status_code == requests.codes.ok:
            records = index_req.json().get('records')
        else:
            records = {
                "code": index_req.json().get('resultcode'),
                "msg": "cninfo接口错误：" + index_req.json().get('resultmsg')
            }
        
        return records
        
    def getIndustry(self, scode="000001"):
        INDUSTRY_API = "/api/bigdata/p_chain_by_scode"
        industry_req = requests.get(self.BASE_URL+INDUSTRY_API, params={
            "scode": scode, 
            "access_token": self.token
        })

        if industry_req.status_code == requests.codes.ok:
            records = industry_req.json().get('records')
        else:
            records = {
                "code": industry_req.json().get('resultcode'),
                "msg": "cninfo接口错误：" + industry_req.json().get('resultmsg')
            }
        
        return records

In [ ]:
stockCode = '000001'
api = CninfoAPI(is_testAPI=True)
industries = api.getIndustry(stockCode)

In [ ]:
industries

In [28]:
industries_df = pd.read_csv("/home/chenhaoran/FinancialTextAnalysis/industries_df.txt", dtype={'COMPANY_CODE': str}, index_col=[0])
industries_df

,UDINDUSTRYNAME,POSITION,CODE,UDINDUSTRYCODE,INDUSTRYCODE,INDUSTRYNAME,NAME,COMPANY_CODE
0,物联网服务,上游,fh009382,I020206,I02020614,智慧工厂,智慧工厂,300316
1,路由器,上游,fh009382,I02010201,I02020614,智慧工厂,智慧工厂,300316
2,大数据技术,上游,fh009382,H020401,I02020614,智慧工厂,智慧工厂,300316
3,自动化检测设备,上游,fh009382,C01081401,I02020614,智慧工厂,智慧工厂,300316
4,计算机网络系统集成,上游,fh009382,H02070103,I02020614,智慧工厂,智慧工厂,300316
...,...,...,...,...,...,...,...,...
148,钙,上游,lsx033n048,B020424,E020143,食品添加剂,护色剂,002568
149,海藻酸钠,上游,lsx033n048,B02050505,E020143,食品添加剂,护色剂,002568
150,左旋肉碱,上游,lsx033n048,E02013108,E020143,食品添加剂,护色剂,002568
151,食品综合,下游,lsx033n048,E0205,E020143,食品添加剂,护色剂,002568


In [23]:
node_dict = dict()
counter = 0
for industry in industries_df.INDUSTRYNAME:
    if node_dict.get(industry) != None:
        # print("seen", industry, counter)
        continue
    else:
        node_dict[industry] = counter
        print(industry, counter)
        counter += 1
print(len(node_dict), counter)
for udindustry in industries_df.UDINDUSTRYNAME:
    if node_dict.get(udindustry) != None:
        continue
    else:
        node_dict[udindustry] = counter
        counter += 1
print(len(node_dict), counter)

智慧工厂 0
机电设备安装劳务 1
橡胶加工设备 2
扩散炉 3
单晶炉 4
气动密封 5
智能能源 6
晶圆 7
多晶炉 8
自动化工程 9
电池片生产设备 10
三极管 11
LED衬底材料 12
新能源乘用车 13
乘用车 14
货车 15
SUV 16
MPV 17
轿车 18
休闲汽车 19
模具 20
加工劳务 21
道路运输 22
工程总承包（EPC） 23
水利工程 24
放射性金属矿石 25
电力设计 26
电力工程施工 27
机电工程 28
房地产开发 29
水力发电 30
工程勘察设计 31
水电机组及相关设备 32
股份制银行 33
信用贷款 34
跨境支付 35
消费金融 36
区块链金融 37
中小微企业贷 38
信用卡 39
供应链金融 40
直销银行 41
信贷 42
企业金融 43
传统仓储 44
度假型公寓 45
办公楼 46
养老地产 47
住宅租赁 48
nan 49
体育地产 50
园区物业管理 51
商场开发 52
普通住宅 53
人才安居房 54
物流地产 55
居住物业管理 56
商业物业管理 57
办公用房建筑工程 58
建筑防水工程 59
第三方工程评估 60
物业管理 61
军用声纳 62
网络安全 63
服务器 64
开关电源 65
安全集成 66
信息加密/身份认证 67
通信电源 68
液晶电视（含CCFL及LED电视） 69
电源 70
打印机及其零部件 71
台式电脑 72
一体机 73
导弹防御系统 74
电子对抗系统 75
LED液晶显示器 76
网络安全设备 77
协同管理软件 78
计量检测 79
信息技术 80
信息安全 81
涤纶加弹丝 82
涤纶纤维(聚酯纤维) 83
涤纶牵伸丝 84
化学纤维 85
石化服务 86
热电、供暖 87
智能制造 88
一次电池 89
氢燃料电池 90
石墨烯及其制品 91
消费电池 92
锂电结构件 93
电池 94
锂电池 95
电池模组及电池系统 96
电表 97
自动光学检测设备AOI 98
锰酸锂 99
磷酸铁锂 100
三元材料 101
磷酸铁锂电池 102
三元锂电池 103
储能设备 104
电子元器件 105
生产平台设备 106
油气设备 107
钻井平台设备 108
作业船 109
船舶制造 110
船舶维护服务 111
航空母舰 11

In [25]:
# (产业链A, 上游, 产业链B)
edge_dict = dict()
counter = 0
for row in industries_df.iterrows():
    edge_key = ''
    ind_1 = row[1]['UDINDUSTRYNAME']
    ind_2 = row[1]['INDUSTRYNAME']
    rel = row[1]['POSITION']
    if rel == '上游':
        edge_key = str(node_dict.get(ind_2)) + '-' + str(node_dict.get(ind_1))
    else:
        edge_key = str(node_dict.get(ind_1)) + '-' + str(node_dict.get(ind_2))
    if edge_dict.get(edge_key) != None:
        continue
    else:
        edge_dict[edge_key] = counter
        counter += 1
print(len(edge_dict))

28388


In [ ]:
with open('/home/chenhaoran/FinancialTextAnalysis/CSI300_industry_graph/entity2id.txt', 'w') as f:
    f.write(str(len(node_dict)) + '\n')
    for (key, value) in node_dict.items():
        # print(key, value)
        f.write(str(key) + '\t' + str(value) + '\n')

In [ ]:
with open('/home/chenhaoran/FinancialTextAnalysis/CSI300_industry_graph/relation2id.txt', 'w') as f:
    f.write("1" + '\n')
    f.write('上游' + '\t0\n')

In [28]:
with open('/home/chenhaoran/FinancialTextAnalysis/CSI300_industry_graph/train2id.txt', 'w') as f:
    f.write(str(len(edge_dict)) + '\n')
    for key in edge_dict.keys():
        head, tail = key.split('-')
        f.write(head + '\t' + tail + '\t0\n')

## 构建产业名称到embedding的字典

In [1]:
import pickle
import json
import pandas as pd
import numpy as np

In [2]:
with open('/home/chenhaoran/GraphEmbedding/OpenKE/checkpoint/transh.embed.vec') as f:
    inds_embeddings = json.load(f)
inds_embeddings = inds_embeddings.get('ent_embeddings.weight')

In [4]:
len(inds_embeddings), len(inds_embeddings[0])

(6527, 200)

In [16]:
emb_dict = {}
with open('/home/chenhaoran/FinancialTextAnalysis/CSI300_industry_graph/entity2id.txt', 'r') as f:
    for l in f.readlines()[1:]:
        line = l.strip()
        name, idx = line.split('\t')
        emb_dict[name] = inds_embeddings[int(idx)]

In [18]:
'''
shape: 6527 x 200
{
    '智慧工厂': [],
    ...
}
'''
with open('/home/chenhaoran/FinancialTextAnalysis/coling2018/csi300_industry_graph/industry_embeddings.dict', 'w') as f:
    f.write(str(emb_dict))

In [3]:
with open('/home/chenhaoran/FinancialTextAnalysis/listed_companies_info.dump', 'rb') as f:
    listed_companies_info = pickle.load(f)
listed_companies_info

,Cuntrycd,Stkcd,Conme,Conme_en,Indcd,Indnme,Nindcd,Nindnme,Nnindcd,Nnindnme,...,Crcd,Statdt,Commnt,Markettype,PROVINCE,PROVINCECODE,CITY,CITYCODE,OWNERSHIPTYPE,OWNERSHIPTYPECODE
Stknme,,,,,,,,,,,,,,,,,,,,,
平安银行,10,000001,平安银行股份有限公司,"Ping An Bank Co., Ltd.",0001,金融,I01,银行业,J66,货币金融服务,...,NaN,2021-08-27,NaN,4.0,广东省,440000,深圳市,440300,私营企业,P0306
万科A,10,000002,万科企业股份有限公司,"China Vanke Co., Ltd.",0003,房地产,J,房地产业,K70,房地产业,...,NaN,2021-08-27,02202,4.0,广东省,440000,深圳市,440300,国营或国有控股,P0301
PT 金田A,10,000003,金田实业(集团)股份有限公司,"Gintian Industry (Group) Co., Ltd.",0004,综合,M,综合类,S90,综合,...,NaN,2002-06-14,NaN,4.0,广东省,440000,深圳市,440300,国营或国有控股,P0301
国华网安,10,000004,深圳国华网安科技股份有限公司,"Shenzhen Cau Technology Co., Ltd.",0002,公用事业,G87,计算机应用服务业,I65,软件和信息技术服务业,...,NaN,2021-08-27,NaN,4.0,广东省,440000,深圳市,440300,私营企业,P0306
ST 星源,10,000005,深圳世纪星源股份有限公司,Shenzhen Fountain Corporation,0002,公用事业,K99,其他社会服务业,N77,生态保护和环境治理业,...,NaN,2021-08-27,NaN,4.0,广东省,440000,深圳市,440300,中外合资,P0303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
锦港B股,10,900952,锦州港股份有限公司,"Jinzhou Port Co., Ltd.",0002,公用事业,F07,水上运输业,G55,水上运输业,...,600190,2021-08-27,NaN,2.0,辽宁省,210000,锦州市,210700,国营或国有控股,P0301
凯马B,10,900953,恒天凯马股份有限公司,"Kama Co., Ltd.",0005,工业,C75,交通运输设备制造业,C36,汽车制造业,...,NaN,2021-08-27,NaN,2.0,上海市,310000,上海市,310000,国营或国有控股,P0301
*ST 海创B,10,900955,海航创新股份有限公司,"HNA Innovation Co., Ltd.",0003,房地产,J,房地产业,K70,房地产业,...,600555,2021-08-27,NaN,2.0,海南省,460000,三亚市,460200,中外合资,P0303


In [19]:
code_to_name = listed_companies_info['Stkcd'].reset_index().rename(columns={'Stkcd': 'COMPANY_CODE'})
code_to_name

,Stknme,COMPANY_CODE
0,平安银行,000001
1,万科A,000002
2,PT 金田A,000003
3,国华网安,000004
4,ST 星源,000005
...,...,...
4862,锦港B股,900952
4863,凯马B,900953
4864,*ST 海创B,900955
4865,东贝B股,900956


In [36]:
name_to_industries = industries_df.join(code_to_name.set_index('COMPANY_CODE'), on="COMPANY_CODE", lsuffix='_ind', rsuffix='_std', how='inner')
name_to_industries = name_to_industries[['Stknme', 'INDUSTRYNAME']].drop_duplicates()
name_to_industries

,Stknme,INDUSTRYNAME
0,晶盛机电,智慧工厂
32,晶盛机电,机电设备安装劳务
78,晶盛机电,橡胶加工设备
105,晶盛机电,扩散炉
109,晶盛机电,单晶炉
...,...,...
367,君正集团,水泥
0,百润股份,食用香精
6,百润股份,鸡尾酒
15,百润股份,香料香精


In [38]:
name_to_industries_dict = {}
for row in name_to_industries.iterrows():
    stknme = row[1]['Stknme']
    indnme = row[1]['INDUSTRYNAME']
    ind_list = name_to_industries_dict.get(stknme)
    if ind_list:
        ind_list.append(indnme)
    else:
        name_to_industries_dict[stknme] = [indnme]
len(name_to_industries_dict)


297

In [39]:
'''
length: 297
[
    {'晶盛机电': ['智慧工厂', '机电设备安装劳务', '橡胶加工设备', '扩散炉', '单晶炉', '气动密封', '智能能源', '晶圆', '多晶炉', '自动化工程', '电池片生产设备', '三极管', 'LED衬底材料'] },
    ...
]
'''
with open('/home/chenhaoran/FinancialTextAnalysis/coling2018/csi300_industry_graph/name_to_industries.dict', 'w') as f:
    f.write(str(name_to_industries_dict))

## 测试industry encoder torch module

In [1]:
from turtle import forward
import torch
import torch.nn as nn
import torch.nn.functional as F
# from torch.autograd import Variable
from torch.nn import LSTM

In [2]:
def sequence_mask(lengths, max_len = None, device = 0):
	device = torch.device("cuda:%s" % device\
	 if torch.cuda.is_available() and device != -1 else "cpu")
	if not max_len:
		max_len = torch.max(lengths)
	#(1, max_len)
	range_matrix = torch.arange(0, max_len, 1, \
		dtype = torch.float, device = device).unsqueeze(0)
	#(batch_size, 1)
	lengths_expanded = lengths.float().unsqueeze(-1)
	return torch.lt(range_matrix, lengths_expanded).float()

def softmax_with_mask(score, mask = None, dim = -1):
	if mask is not None:
		score = score.masked_fill(mask == 0., -1e9)
	p_attn = F.softmax(score, dim = dim)
	return p_attn

In [3]:
class AttNet(nn.Module):
  """docstring for AttNet"""
  def __init__(self, config, input_dim):
    super(AttNet, self).__init__()
    self.config = config
    # self.affine = nn.Linear(input_dim, 1)
    self.w_q = nn.Linear(input_dim, input_dim, bias = False)
    self.w_f = nn.Linear(input_dim, input_dim, bias = True)

    self.affine = nn.Sequential(nn.Tanh(),\
     nn.Dropout(p = config.dropout), nn.Linear(input_dim, 1))

  #query(batch_size, 1, hidden_dim)
  #facts(batch_size, max_seqlen, hidden_dim)
  #fact_mask(batch_size, max_seqlen, 1)
  #return (batch_size, hidden_dim)
  def forward(self, facts, query = None, fact_mask = None):

    if query is not None:
      # max_seqlen = facts.size(1)
      # query = query.unsqueeze(1).repeat(1, max_seqlen, 1)
      # #(batch_size, max_seqlen, hidden_dim * 2)
      # output = torch.cat([query, facts], dim = -1)
      # print("baseline.py line 33, AttNet query.shape: ", query.shape, " facts.shape: ", facts.shape)
      output = self.w_q(query.unsqueeze(1)) + self.w_f(facts)
    else:
      output = facts

    #(batch_size, max_seqlen, 1)
    weights = self.affine(output)
    norm_weights = softmax_with_mask(weights, mask = fact_mask, dim = 1)

    return torch.sum(norm_weights * facts, dim = 1)


class Attention(nn.Module):

  def __init__(self, config, input_dim):
    super(Attention, self).__init__()

    self.att = nn.Sequential(\
      nn.Linear(input_dim, input_dim), nn.Tanh(),\
      nn.Linear(input_dim, 1, bias = False))
    self.dropout = nn.Dropout(p = config.dropout)

  #facts(batch_size, max_seq_len, input_dim)
  #fact_mask(batch_size, max_seq_len, 1)
  #return(batch_size, embed_dim)
  def forward(self, facts, fact_mask = None, hard = False, tau = 1, dim = -1):

    #(batch_size, max_seq_len, 1)
    scores = self.att(facts)

    if hard:
      scores = scores.squeeze(-1)
      if fact_mask is not None:
        p_attn = gumbel_softmax(scores, fact_mask.squeeze(-1), tau = tau)
      else:
        p_attn = gumbel_softmax(scores, None, tau = tau)
      return torch.sum(p_attn.unsqueeze(-1) * facts, dim = dim)

    else:
      p_attn = softmax_with_mask(scores, fact_mask, dim = dim)
      p_attn = self.dropout(p_attn)
      return torch.sum(p_attn * facts, dim = dim)

#sequential representation
class SequentialRepr(nn.Module):
  def __init__(self, config, input_dim = 100, mode = 'cnn'):
    super(SequentialRepr, self).__init__()

    self.mode = mode
    self.config = config
    self.input_dim = input_dim

    self.c0_initer = nn.Linear(input_dim, config.hidden_dim)
  
    if mode == 'cnn':
      ks = [int(c) for c in config.kernel_sizes.split(',')]
      self.encoder = CNNText(config, ks)

    elif mode.startswith("gru"):
      self.encoder = nn.GRU(input_size = input_dim, batch_first = True, \
        hidden_size = config.hidden_dim,  bidirectional = True, \
        num_layers = 1)

    elif mode.startswith('lstm'):
      self.encoder = nn.LSTM(input_size = input_dim, batch_first = True, \
        hidden_size = config.hidden_dim,  bidirectional = True, \
        num_layers = 1)

    if mode.endswith('att'):
      self.att = Attention(config, config.hidden_dim)


  # text_embed(batch_size, seqlen, embed_dim)
  # mask(batch_size, seqlen, 1)
  def forward(self, text_embed, init = None, mask = None):

    hiddens, output = None, None
    if self.mode.startswith("gru") or self.mode.startswith('lstm'):

      if init is not None:

        batch_size = text_embed.size(0) // init.size(0)
        init = c0 = init.unsqueeze(0).repeat(2, batch_size, 1)

        if self.mode.startswith("lstm"):
          c0 = self.c0_initer(c0) if self.config.bert == "true" else c0
          h0 = torch.zeros_like(c0)
          init = (h0, c0)
          # print(">>>>>", text_embed.shape, h0.shape, c0.shape)
      hiddens, output = self.encoder(text_embed, init)
      hsize = hiddens.size()
      hiddens = hiddens.view(*hsize[:-1], 2, -1)
      hiddens = hiddens.mean(dim = -2, keepdim = False)
      # print("baseline.py line 126 hiddens.shape==>", hiddens.shape)
      if self.mode.endswith('avg'):
        if mask is not None:
          output = torch.sum(hiddens * mask, dim = 1) / (torch.sum(mask, dim = 1) + 1e-8)
        else:
          output = torch.mean(hiddens, dim = 1, keepdim = False)
      elif self.mode.endswith('att'):
        output = self.att(hiddens, mask, dim = 1)
      # print("baseline.py line 134 output.shape==>", output.shape)
    elif self.mode == 'cnn':
      output = self.encoder(text_embed)

    else:
      raise Exception('mode %s not defined!' % self.mode)
    return hiddens, output

In [4]:
class IndustryEncoder(nn.Module):
  def __init__(self, config, dim = 128,) -> None:
    super(IndustryEncoder, self).__init__()
    self.ind_dim = dim
    self.target_to_inds = config.target_to_inds
    self.inds_embs = config.inds_embs
    self.seq_encoder = SequentialRepr(config, input_dim = config.embed_dim, mode = 'lstm')
    self.inds_attn = AttNet(config, input_dim=128)
    self.linear = nn.Linear(config.embed_dim, self.ind_dim)

  def forward(self, target, lead):
    inds = self.target_to_inds
    _, lead_reprs = self.seq_encoder(lead, )
    query = self.linear(lead_reprs)
    industry_reprs = self.inds_attn()
    return 
